In [ ]:
import pandas as pd

In [ ]:
all_matches = pd.DataFrame()
seasons = ['2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']

for season in seasons:
    all_matches = pd.concat([all_matches, pd.read_csv(f'../../data/Advanced/pl{season}.csv')])
    all_matches['season'] = season

all_matches.reset_index(drop=True, inplace=True)

In [ ]:
all_matches = all_matches[[
    "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HS", "AS", "HST", "AST",
    "HF", "AF", "HC", "AC",
    "HY", "AY", "HR", "AR",
    "B365H", "B365D", "B365A"
]]


In [ ]:
rename_map = {
    "Date": "date",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR":  "result",

    "HTHG": "home_ht_goals",
    "HTAG": "away_ht_goals",
    "HTR":  "ht_result",

    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_sot",
    "AST": "away_sot",
    "HF": "home_fouls",
    "AF": "away_fouls",
    "HC": "home_corners",
    "AC": "away_corners",
    "HY": "home_yellow",
    "AY": "away_yellow",
    "HR": "home_red",
    "AR": "away_red",

    "B365H": "odds_home_win",
    "B365D": "odds_draw",
    "B365A": "odds_away_win"
}
all_matches.rename(columns=rename_map, inplace=True)

In [ ]:
all_matches["date"] = pd.to_datetime(all_matches["date"], dayfirst=True)
all_matches.sort_values(by="date", inplace=True)

In [ ]:
all_matches["result"] = all_matches["result"].map({"H": 2, "D": 0, "A": 1})

In [ ]:
all_matches.head().to_clipboard(index=False)

In [ ]:
import pandas as pd

def add_rolling_team_stats(df, window=5):

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    stat_map = {
        "goals_scored": ("home_goals", "away_goals"),
        "goals_conceded": ("away_goals", "home_goals"),
        "shots": ("home_shots", "away_shots"),
        "shots_on_target": ("home_sot", "away_sot"),
        "fouls": ("home_fouls", "away_fouls"),
        "corners": ("home_corners", "away_corners"),
        "yellow_cards": ("home_yellow", "away_yellow"),
        "red_cards": ("home_red", "away_red"),
    }

    # Build long format
    home_df = df.rename(columns={v[0]: k for k, v in stat_map.items()})
    away_df = df.rename(columns={v[1]: k for k, v in stat_map.items()})

    home_df["team"] = home_df["home_team"]
    away_df["team"] = away_df["away_team"]

    cols = ["date", "team"] + list(stat_map.keys())
    long_df = pd.concat([home_df[cols], away_df[cols]], ignore_index=True)
    long_df = long_df.sort_values(["team", "date"]).reset_index(drop=True)

    # Compute rolling stats (fixed)
    for stat in stat_map.keys():
        long_df[f"{stat}_rolling_avg"] = (
            long_df.groupby("team")[stat]
                   .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )

    # HOME merge
    home_merge_cols = {f"{stat}_rolling_avg": f"home_{stat}_rolling"
                       for stat in stat_map.keys()}

    df = df.merge(
        long_df[["date", "team"] + list(home_merge_cols.keys())],
        left_on=["date", "home_team"],
        right_on=["date", "team"],
        how="left"
    ).drop(columns=["team"])
    df = df.rename(columns=home_merge_cols)

    # AWAY merge
    away_merge_cols = {f"{stat}_rolling_avg": f"away_{stat}_rolling"
                       for stat in stat_map.keys()}

    df = df.merge(
        long_df[["date", "team"] + list(away_merge_cols.keys())],
        left_on=["date", "away_team"],
        right_on=["date", "team"],
        how="left"
    ).drop(columns=["team"])
    df = df.rename(columns=away_merge_cols)

    # Fill NAN
    df = df.fillna(0)

    return df


In [ ]:
match_stats = add_rolling_team_stats(all_matches, window=5)

In [ ]:
match_stats.info()

In [ ]:
import numpy as np
import pandas as pd

def expected_result(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(elo_a, elo_b, score_a, score_b, k=20):
    expected_a = expected_result(elo_a, elo_b)
    expected_b = 1 - expected_a

    if score_a > score_b:   # home win
        actual_a, actual_b = 1, 0
    elif score_a < score_b: # away win
        actual_a, actual_b = 0, 1
    else:
        actual_a, actual_b = 0.5, 0.5

    new_a = elo_a + k * (actual_a - expected_a)
    new_b = elo_b + k * (actual_b - expected_b)

    return new_a, new_b

def add_elo_features(df):
    df = df.sort_values("date").copy()

    team_elos = {}
    base_elo = 1500

    df["home_elo_before"] = 0.0
    df["away_elo_before"] = 0.0

    for i, row in df.iterrows():
        home, away = row["home_team"], row["away_team"]

        # If new team, initialize
        team_elos.setdefault(home, base_elo)
        team_elos.setdefault(away, base_elo)

        # Assign ELO before match
        df.at[i, "home_elo_before"] = team_elos[home]
        df.at[i, "away_elo_before"] = team_elos[away]

        # Update after match
        new_home, new_away = update_elo(
            team_elos[home], team_elos[away],
            row["home_goals"], row["away_goals"]
        )

        team_elos[home] = new_home
        team_elos[away] = new_away

    return df


In [ ]:
match_stats = add_elo_features(match_stats)

In [ ]:
match_stats.head()

In [ ]:
def add_bookmaker_features(df):
    df = df.copy()

    # Convert to implied probabilities
    df["prob_home"] = 1 / df["odds_home_win"]
    df["prob_draw"] = 1 / df["odds_draw"]
    df["prob_away"] = 1 / df["odds_away_win"]

    # Normalize to remove overround
    total = df[["prob_home", "prob_draw", "prob_away"]].sum(axis=1)

    df["prob_home"] /= total
    df["prob_draw"] /= total
    df["prob_away"] /= total

    return df


In [ ]:
match_stats = add_bookmaker_features(match_stats)

In [ ]:
match_stats

In [ ]:
final_df = match_stats.drop(columns=["home_goals", "away_goals",
        "home_shots", "away_shots",
        "home_sot", "away_sot",
        "home_fouls", "away_fouls",
        "home_corners", "away_corners",
        "home_yellow", "away_yellow",
        "home_red", "away_red"])

In [ ]:
final_df = pd.get_dummies(final_df, columns=["home_team", "away_team"], drop_first=True)

In [ ]:
final_df.to_csv('../../data/Advanced/pl_all_seasons_enhanced.csv', index=False)

In [ ]:
more_data = pd.read_clipboard()
more_data

In [ ]:
more_data = pd.concat([more_data, pd.read_clipboard()])
more_data

In [ ]:
more_data = more_data[more_data["Home"] != 'Home']

In [ ]:
more_data.info()

In [ ]:
more_data.rename(columns={"Home": "home_team", "Away": "away_team", 'Date': 'date'}, inplace=True)

In [ ]:
more_data.drop(columns=["Score"], inplace=True)

In [ ]:
more_data.info()

In [ ]:
final_df['home_team'].sort_values().value_counts()

In [ ]:
more_data['home_team'].sort_values().value_counts()

In [ ]:
team_name_map = {
        "Newcastle Utd": "Newcastle",
        "Manchester Utd": "Man United",
        "Manchester City": "Man City",
        "Leicester City": "Leicester",
        "Leeds United": "Leeds",
        "Ipswich Town": "Ipswich",
        "Luton Town": "Luton",
        "Norwich City": "Norwich",
        "Sheffield Utd": "Sheffield United",
        "Nott'ham Forest": "Nott'm Forest"
    }

more_data['home_team'] = more_data['home_team'].replace(team_name_map)
more_data['away_team'] = more_data['away_team'].replace(team_name_map)

In [ ]:
final_df['date'] = pd.to_datetime(final_df['date'])
more_data['date'] = pd.to_datetime(more_data['date'])

In [ ]:
merged = final_df.merge(more_data, on=['date', 'home_team', 'away_team'], how='left')

In [ ]:
merged.columns

In [ ]:
merged.drop(columns=['Wk'], inplace=True)

In [ ]:
merged.to_csv('../../data/Advanced/pl_final.csv', index=False)